# DINOv2 vs DINOv3: what does art training do to each?

Runs every analysis for every setting, then shows each result side by side:

| Setting | Model | Blocks trained |
|---|---|---|
| 1 | DINOv2-small | last 4 |
| 2 | DINOv2-small | all 12 |
| 3 | DINOv3-small | last 4 |

**How it works:** models are processed one at a time and freed afterwards (keeps memory low). Seed 0 drives every figure; extra seeds feed the robustness chart.

**Before running:**
- Colab: Runtime → Change runtime type → **T4 GPU** (High-RAM helps). Kaggle: accelerator **GPU T4 x2**, Internet **on**.
- DINOv3 is gated: request access on its Hugging Face page, then add your token as a secret named `HF_TOKEN` (Colab: key icon, notebook access on; Kaggle: see the cell below). Without access, DINOv3 is skipped and everything else still runs.
- Run all cells top to bottom. Expect roughly 25–40 minutes on a T4.

In [ ]:
!pip -q install datasets umap-learn "transformers>=4.56"

In [ ]:
import os
os.environ["USE_TF"] = "0"   # stop transformers loading TensorFlow (saves memory)

import copy, gc, math, random, time
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from PIL import Image
from torchvision import transforms as T
from datasets import load_dataset
from transformers import AutoModel
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from scipy.stats import rankdata
import umap

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print("WARNING: no GPU attached. Switch to a T4 GPU runtime, or this will be very slow.")

**Kaggle only:** uncomment to load your Hugging Face token from Add-ons → Secrets (Colab loads `HF_TOKEN` automatically).

In [ ]:
# from kaggle_secrets import UserSecretsClient
# os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

## Settings

In [ ]:
SETTINGS = [
    dict(model_id="facebook/dinov2-small", n_blocks=4),
    dict(model_id="facebook/dinov2-small", n_blocks=12),
    dict(model_id="facebook/dinov3-vits16-pretrain-lvd1689m", n_blocks=4),
]
SEEDS = [0, 1, 2]        # seed 0 drives the figures; all seeds feed the robustness chart
EPOCHS = 5
LR_BACKBONE = 2e-5

STYLES = ["Impressionism", "Baroque", "Cubism", "Ukiyo_e", "Pop_Art",
          "Abstract_Expressionism", "Realism", "Color_Field_Painting"]
PER_STYLE = 120
MAX_SCAN = 25000

ABSTRACT = ["Abstract_Expressionism", "Color_Field_Painting"]
REPRESENTATIONAL = ["Baroque", "Realism", "Impressionism", "Ukiyo_e"]

## Load paintings

Same WikiArt sample for every model: 120 paintings per style, 80% for training and 20% for testing.

In [ ]:
torch.manual_seed(0); np.random.seed(0); random.seed(0)
ds = load_dataset("huggan/wikiart", split="train", streaming=True)
if ds.features is None:
    ds = ds._resolve_features()
all_styles = ds.features["style"].names
style_ids = {all_styles.index(s): s for s in STYLES if s in all_styles}
id_to_label = {sid: i for i, sid in enumerate(style_ids)}
class_names = [style_ids[sid] for sid in style_ids]

counts = {sid: 0 for sid in style_ids}
images, labels = [], []
for i, ex in enumerate(ds.shuffle(seed=0, buffer_size=1000)):
    sid = ex["style"]
    if sid in counts and counts[sid] < PER_STYLE:
        img = ex["image"].convert("RGB")
        img.thumbnail((320, 320))
        images.append(img); labels.append(id_to_label[sid]); counts[sid] += 1
    if all(c >= PER_STYLE for c in counts.values()) or i >= MAX_SCAN:
        break
labels = np.array(labels)
del ds; gc.collect()

tr_idx, te_idx = train_test_split(np.arange(len(images)), test_size=0.2, stratify=labels, random_state=0)
test_imgs = [images[j] for j in te_idx]
examples_pos = [int(np.where(labels[te_idx] == k)[0][0]) for k in range(min(5, len(class_names)))]
ia_all = np.where(np.isin(labels, [class_names.index(n) for n in ABSTRACT if n in class_names]))[0]
ir_all = np.where(np.isin(labels, [class_names.index(n) for n in REPRESENTATIONAL if n in class_names]))[0]

print(f"Scanned {i + 1} images, kept {len(images)}: {len(tr_idx)} train / {len(te_idx)} test")
for sid, c in counts.items():
    print(f"  {style_ids[sid]:<24} {c}")

## Helpers

Measurement functions (no GPU needed), then model functions.

In [ ]:
def short(model_id):
    return model_id.split("/")[-1].replace("-pretrain-lvd1689m", "")

def setting_name(s):
    return f"{short(s['model_id'])} · {s['n_blocks']} blocks"

def before_name(model_id):
    return f"{short(model_id)} · before"

def show_img(img):
    """Same framing the models see: shorter side to 256, centre crop 224."""
    w, h = img.size
    sc = 256 / min(w, h)
    img = img.resize((max(256, round(w * sc)), max(256, round(h * sc))), Image.BICUBIC)
    w, h = img.size
    l, t = (w - 224) // 2, (h - 224) // 2
    return img.crop((l, t, l + 224, t + 224))

def probe(emb):
    sc = StandardScaler().fit(emb[tr_idx])
    clf = LogisticRegression(max_iter=3000).fit(sc.transform(emb[tr_idx]), labels[tr_idx])
    return clf.score(sc.transform(emb[te_idx]), labels[te_idx])

def split_ratio(emb, ia, ir):
    """Distance between abstract and representational centres / average spread within each."""
    e = emb / np.linalg.norm(emb, axis=1, keepdims=True)
    A, R_ = e[ia], e[ir]
    ca, cr = A.mean(0), R_.mean(0)
    between = 1 - ca @ cr / (np.linalg.norm(ca) * np.linalg.norm(cr))
    within = np.mean([np.mean(1 - X @ c / np.linalg.norm(c)) for X, c in ((A, ca), (R_, cr))])
    return between / within

def split_ci(emb, n=500, seed=0):
    rng = np.random.default_rng(seed)
    vals = [split_ratio(emb, rng.choice(ia_all, len(ia_all)), rng.choice(ir_all, len(ir_all))) for _ in range(n)]
    return np.percentile(vals, [2.5, 97.5])

def linear_cka(X, Y):
    X = X.astype(np.float64) - X.mean(0); Y = Y.astype(np.float64) - Y.mean(0)
    return np.linalg.norm(Y.T @ X) ** 2 / (np.linalg.norm(X.T @ X) * np.linalg.norm(Y.T @ Y))

PROP_NAMES = ["brightness", "contrast", "saturation", "colourfulness", "warmth", "edge density"]
def image_props(img):
    a = np.asarray(show_img(img)).astype(np.float32) / 255
    r, g, b = a[..., 0], a[..., 1], a[..., 2]
    lum = 0.299 * r + 0.587 * g + 0.114 * b
    mx, mn = a.max(-1), a.min(-1)
    sat = (mx - mn) / (mx + 1e-6)
    rg, yb = r - g, 0.5 * (r + g) - b
    colourful = np.hypot(rg.std(), yb.std()) + 0.3 * np.hypot(rg.mean(), yb.mean())
    gy, gx = np.gradient(lum)
    return [lum.mean(), lum.std(), sat.mean(), colourful, (r - b).mean(), np.mean(np.hypot(gx, gy) > 0.05)]

def make_pattern(symmetric, n_on=20, rng=None):
    if symmetric:
        half = np.zeros(32); half[rng.choice(32, n_on // 2, replace=False)] = 1
        left = half.reshape(8, 4)
        grid = np.concatenate([left, left[:, ::-1]], axis=1)
    else:
        flat = np.zeros(64); flat[rng.choice(64, n_on, replace=False)] = 1
        grid = flat.reshape(8, 8)
    pattern = ((1 - grid) * 255).astype(np.uint8).repeat(28, 0).repeat(28, 1)
    canvas = np.full((256, 256), 255, np.uint8)
    canvas[16:240, 16:240] = pattern
    return Image.fromarray(canvas).convert("RGB")

def upsample(a, size=224):
    a = np.asarray(Image.fromarray(a.astype(np.float32)).resize((size, size), Image.BILINEAR))
    return (a - a.min()) / (a.max() - a.min() + 1e-8)

_rng = np.random.default_rng(0)
SYM = [make_pattern(True, rng=_rng) for _ in range(100)]
ASYM = [make_pattern(False, rng=_rng) for _ in range(100)]

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
eval_tf = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize(MEAN, STD)])
train_tf = T.Compose([T.RandomResizedCrop(224, scale=(0.6, 1.0)), T.RandomHorizontalFlip(),
                      T.ToTensor(), T.Normalize(MEAN, STD)])

def load_backbone(model_id):
    return AutoModel.from_pretrained(model_id, attn_implementation="eager").to(device).eval()

@torch.no_grad()
def embed(backbone, imgs, bs=64):
    """CLS-token embedding per image."""
    backbone.eval()
    out = []
    for i in range(0, len(imgs), bs):
        px = torch.stack([eval_tf(im) for im in imgs[i:i + bs]]).to(device)
        out.append(backbone(pixel_values=px).last_hidden_state[:, 0].float().cpu())
    return torch.cat(out).numpy()

def get_blocks(bb):
    """Transformer blocks (named differently in DINOv2 and DINOv3)."""
    for path in ("encoder.layer", "model.layer", "layer"):
        obj = bb
        try:
            for name in path.split("."):
                obj = getattr(obj, name)
        except AttributeError:
            continue
        if isinstance(obj, nn.ModuleList):
            return obj
    raise ValueError("Could not find the transformer blocks")

def get_final_norm(bb):
    return bb.layernorm if hasattr(bb, "layernorm") else bb.norm

class StyleClassifier(nn.Module):
    def __init__(self, backbone, n_classes):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(backbone.config.hidden_size, n_classes)
    def forward(self, px):
        return self.head(self.backbone(pixel_values=px).last_hidden_state[:, 0])

def batches(index, tf, bs=32, shuffle=False):
    index = np.random.permutation(index) if shuffle else index
    for i in range(0, len(index), bs):
        b = index[i:i + bs]
        yield torch.stack([tf(images[j]) for j in b]).to(device), torch.tensor(labels[b]).to(device)

@torch.no_grad()
def accuracy(model, index):
    model.eval()
    return sum((model(x).argmax(1) == y).sum().item() for x, y in batches(index, eval_tf)) / len(index)

def finetune(backbone, seed=0, n_blocks=4, epochs=EPOCHS, lr=LR_BACKBONE, on_epoch=None):
    """Art training: copy the backbone, unfreeze its last n_blocks, train on style labels."""
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    model = StyleClassifier(copy.deepcopy(backbone), len(class_names)).to(device)
    for p in model.backbone.parameters():
        p.requires_grad = False
    blocks = get_blocks(model.backbone)
    for block in (blocks[max(len(blocks) - n_blocks, 0):] if n_blocks > 0 else []):
        for p in block.parameters():
            p.requires_grad = True
    for p in get_final_norm(model.backbone).parameters():
        p.requires_grad = True
    opt = torch.optim.AdamW([
        {"params": [p for p in model.backbone.parameters() if p.requires_grad], "lr": lr},
        {"params": model.head.parameters(), "lr": 1e-3}], weight_decay=0.05)
    for epoch in range(epochs):
        model.train()
        for x, y in batches(tr_idx, train_tf, shuffle=True):
            loss = F.cross_entropy(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step()
        if on_epoch is not None:
            on_epoch(epoch + 1, model)
    return model

@torch.no_grad()
def layer_reps(backbone, imgs, bs=64):
    """Mean-pooled token representation at every layer (None if the model won't return them)."""
    try:
        backbone.eval()
        per_layer = None
        for i in range(0, len(imgs), bs):
            px = torch.stack([eval_tf(im) for im in imgs[i:i + bs]]).to(device)
            hs = backbone(pixel_values=px, output_hidden_states=True).hidden_states
            means = [h.mean(1).float().cpu() for h in hs]
            per_layer = means if per_layer is None else [torch.cat([a, b]) for a, b in zip(per_layer, means)]
        return [p.numpy() for p in per_layer]
    except Exception as err:
        print(f"    layer representations unavailable: {type(err).__name__}: {str(err)[:120]}")
        return None

@torch.no_grad()
def attention_maps(backbone, imgs, bs=32):
    """CLS-to-patch attention, every layer and head: (images, layers, heads, side, side), float16."""
    try:
        backbone.eval()
        out = []
        for i in range(0, len(imgs), bs):
            px = torch.stack([eval_tf(im) for im in imgs[i:i + bs]]).to(device)
            atts = backbone(pixel_values=px, output_attentions=True).attentions
            maps = []
            for a in atts:                                  # (B, heads, tokens, tokens)
                cls = a[:, :, 0, 1:].float()
                side = int(math.sqrt(cls.shape[-1]))
                cls = cls[..., -side * side:]               # patch tokens only (drops register tokens)
                cls = cls / cls.sum(-1, keepdim=True)
                maps.append(cls.reshape(cls.shape[0], cls.shape[1], side, side))
            out.append(torch.stack(maps, 1).half().cpu().numpy())
        return np.concatenate(out)
    except Exception as err:
        print(f"    attention maps unavailable: {type(err).__name__}: {str(err)[:120]}")
        return None

@torch.no_grad()
def weight_change(before, after):
    pb, pa = dict(before.named_parameters()), dict(after.named_parameters())
    groups = {}
    for name, p0 in pb.items():
        parts = name.split(".")
        k = next((i for i in range(len(parts) - 1) if parts[i] == "layer" and parts[i + 1].isdigit()), None)
        g = f"block {parts[k + 1]}" if k is not None else parts[0]
        d, n = groups.get(g, (0.0, 0.0))
        groups[g] = (d + (pa[name] - p0).float().pow(2).sum().item(), n + p0.float().pow(2).sum().item())
    return {g: np.sqrt(d) / (np.sqrt(n) + 1e-12) for g, (d, n) in groups.items()}

def typicality(backbone, art_emb):
    centroid = art_emb[tr_idx].mean(0)
    def typ(pats):
        e = embed(backbone, pats)
        return e @ centroid / (np.linalg.norm(e, axis=1) * np.linalg.norm(centroid))
    return dict(sym=typ(SYM), asym=typ(ASYM))

## Run everything

For each model: measure the frozen ("before") version, then art-train each setting with every seed. Progress prints as it goes.

In [ ]:
R = {"before": {}, "after": {}, "seeds": [], "failed": []}
t0 = time.time()
clock = lambda: f"[{(time.time() - t0) / 60:.1f} min]"

for base_id in dict.fromkeys(s["model_id"] for s in SETTINGS):
    print(f"\n=== {short(base_id)} ===")
    try:
        base = load_backbone(base_id)
    except Exception as err:
        print(f"  could not load, skipping: {type(err).__name__}: {str(err)[:200]}")
        R["failed"].append(base_id)
        continue

    emb0 = embed(base, images)
    R["before"][base_id] = B = dict(
        emb=emb0, probe=probe(emb0), sil=silhouette_score(emb0, labels, metric="cosine"),
        split=split_ratio(emb0, ia_all, ir_all), split_ci=split_ci(emb0),
        layers=layer_reps(base, test_imgs), att=attention_maps(base, test_imgs),
        typ=typicality(base, emb0))
    print(f"  before: probe {B['probe']:.3f}, silhouette {B['sil']:.3f}, split {B['split']:.3f} {clock()}")

    for s in [s for s in SETTINGS if s["model_id"] == base_id]:
        name = setting_name(s)
        traj = [dict(epoch=0, silhouette=B["sil"], split_ratio=B["split"])]
        def record(ep, m):
            e = embed(m.backbone, images)
            traj.append(dict(epoch=ep, silhouette=silhouette_score(e, labels, metric="cosine"),
                             split_ratio=split_ratio(e, ia_all, ir_all), test_acc=accuracy(m, te_idx)))
        for seed in SEEDS:
            m = finetune(base, seed=seed, n_blocks=s["n_blocks"], on_epoch=record if seed == SEEDS[0] else None)
            e1 = embed(m.backbone, images)
            row = dict(setting=name, seed=seed, test_acc=accuracy(m, te_idx), probe=probe(e1),
                       silhouette=silhouette_score(e1, labels, metric="cosine"),
                       split_ratio=split_ratio(e1, ia_all, ir_all))
            R["seeds"].append(row)
            print(f"  {name}, seed {seed}: acc {row['test_acc']:.3f}, probe {row['probe']:.3f}, "
                  f"silhouette {row['silhouette']:.3f}, split {row['split_ratio']:.3f} {clock()}")
            if seed == SEEDS[0]:
                R["after"][name] = dict(
                    base=base_id, n_blocks=s["n_blocks"], emb=e1, traj=traj,
                    test_acc=row["test_acc"], probe=row["probe"], sil=row["silhouette"],
                    split=row["split_ratio"], split_ci=split_ci(e1),
                    layers=layer_reps(m.backbone, test_imgs), att=attention_maps(m.backbone, test_imgs),
                    weights=weight_change(base, m.backbone), typ=typicality(m.backbone, e1))
            del m; gc.collect(); torch.cuda.empty_cache()
    del base; gc.collect(); torch.cuda.empty_cache()

print(f"\nDone {clock()}. Skipped models: {R['failed'] or 'none'}")

## Figures setup

In [ ]:
if not R["after"]:
    raise RuntimeError("No setting finished training; check the messages in the run cell above.")

REP, REPS = {}, []          # every representation: each model before, then its art-trained settings
for bid, B in R["before"].items():
    REPS.append(before_name(bid)); REP[before_name(bid)] = {**B, "kind": "before", "base": bid}
    for name, A in R["after"].items():
        if A["base"] == bid:
            REPS.append(name); REP[name] = {**A, "kind": "after"}
AFTERS = [r for r in REPS if REP[r]["kind"] == "after"]
COL = {r: plt.get_cmap("tab10")(i) for i, r in enumerate(REPS)}
STYLE_CMAP = plt.get_cmap("tab10")
before_of = lambda r: R["before"][REP[r]["base"]]
print("Representations:", *REPS, sep="\n  ")

## 1. Headline numbers

How well each model recognises style, how tightly styles cluster, and how far apart abstract and representational art sit. Art-trained bars show the mean across seeds, with the standard deviation as error bars; "before" bars are single values.

In [ ]:
seeds_df = pd.DataFrame(R["seeds"])
METRICS = ["test_acc", "probe", "silhouette", "split_ratio"]
KEY = {"test_acc": "test_acc", "probe": "probe", "silhouette": "sil", "split_ratio": "split"}
TITLE = {"test_acc": "Style classifier test accuracy", "probe": "Linear probe accuracy",
         "silhouette": "Style cluster tightness (silhouette)",
         "split_ratio": "Abstract vs representational (split ratio)"}

table = pd.DataFrame({r: {m: REP[r].get(KEY[m], np.nan) for m in METRICS} for r in REPS}).T
print("Seed 0 (used in all figures):")
print(table.round(3).to_string())
print(f"\nArt-trained settings across seeds {SEEDS} (mean ± sd):")
agg = seeds_df.groupby("setting")[METRICS].agg(["mean", "std"])
print((agg.xs("mean", axis=1, level=1).round(3).astype(str) + " ± "
       + agg.xs("std", axis=1, level=1).round(3).astype(str)).to_string())

fig, axes = plt.subplots(1, 4, figsize=(19, 4.4))
for ax, m in zip(axes, METRICS):
    for i, r in enumerate(REPS):
        if REP[r]["kind"] == "after":
            vals = seeds_df.loc[seeds_df.setting == r, m]
            v, sd = vals.mean(), (vals.std() if len(vals) > 1 else 0.0)
        else:
            v, sd = REP[r].get(KEY[m], np.nan), 0.0
        if not np.isnan(v):
            ax.bar(i, v, yerr=sd, color=COL[r], capsize=4, alpha=1.0 if REP[r]["kind"] == "after" else 0.45)
    ax.set_xticks(range(len(REPS))); ax.set_xticklabels(REPS, rotation=35, ha="right", fontsize=8)
    ax.set_title(TITLE[m], fontsize=10)
plt.tight_layout(); plt.show()

## 2. Learning curves

How each setting changes epoch by epoch (seed 0). Epoch 0 is before art training. If the split keeps growing after test accuracy stops improving, the reorganisation is driven by memorising labels.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.4))
for r in AFTERS:
    tr = REP[r]["traj"]
    axes[0].plot([t["epoch"] for t in tr], [t["silhouette"] for t in tr], marker="o", color=COL[r], label=r)
    axes[1].plot([t["epoch"] for t in tr], [t["split_ratio"] for t in tr], marker="o", color=COL[r], label=r)
    acc = [t for t in tr if "test_acc" in t]
    axes[2].plot([t["epoch"] for t in acc], [t["test_acc"] for t in acc], marker="o", color=COL[r], label=r)
for ax, t in zip(axes, ["Style cluster tightness (silhouette)", "Abstract vs representational (split ratio)",
                        "Style classifier test accuracy"]):
    ax.set_title(t, fontsize=10); ax.set_xlabel("epoch (0 = before art training)"); ax.grid(alpha=0.3)
axes[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 3. Embedding maps

Each dot is a painting, coloured by style. One row per model: frozen on the left, art-trained settings to the right. Read neighbourhoods, not distances between clusters (UMAP distorts those).

In [ ]:
umap_kw = dict(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=0)
XY = {r: umap.UMAP(**umap_kw).fit_transform(REP[r]["emb"]) for r in REPS}

rows = [[r for r in REPS if REP[r]["base"] == b] for b in R["before"]]
ncol = max(len(row) for row in rows)
fig, axes = plt.subplots(len(rows), ncol, figsize=(5.2 * ncol, 4.8 * len(rows)), squeeze=False)
for i, row in enumerate(rows):
    for j in range(ncol):
        ax = axes[i, j]
        if j >= len(row):
            ax.axis("off"); continue
        r, xy = row[j], XY[row[j]]
        for k, cname in enumerate(class_names):
            msk = labels == k
            ax.scatter(xy[msk, 0], xy[msk, 1], s=6, color=STYLE_CMAP(k), label=cname, alpha=0.8)
        ax.set_title(f"{r}\nsilhouette {REP[r]['sil']:.3f}", fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
handles, labs = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labs, loc="center right", markerscale=3, fontsize=9)
plt.tight_layout(rect=(0, 0, 0.86, 1)); plt.show()

## 4. Thumbnail atlases

The art-trained maps with paintings drawn in place (frame colour = style). Look at what sits on the borders between neighbourhoods.

In [ ]:
def atlas(ax, xy, n=120):
    pick = np.random.RandomState(0).choice(len(images), size=min(n, len(images)), replace=False)
    ax.scatter(xy[:, 0], xy[:, 1], s=1, alpha=0)
    for j in pick:
        thumb = np.asarray(show_img(images[j]).resize((72, 72)))
        ax.add_artist(AnnotationBbox(OffsetImage(thumb, zoom=0.55), xy[j], frameon=True, pad=0.1,
                                     bboxprops=dict(edgecolor=STYLE_CMAP(labels[j]), lw=1.5)))
    ax.set_xticks([]); ax.set_yticks([])

fig, axes = plt.subplots(1, len(AFTERS), figsize=(8 * len(AFTERS), 7.5), squeeze=False)
for ax, r in zip(axes[0], AFTERS):
    atlas(ax, XY[r]); ax.set_title(r, fontsize=12)
plt.tight_layout(); plt.show()

## 5. Features that changed most

For each setting, the 4 embedding features whose spread grew most with art training, and the 6 paintings that activate each most. Compare: do the models develop similar kinds of features?

In [ ]:
FEATS = {}
for r in AFTERS:
    ratio = REP[r]["emb"].var(0) / (before_of(r)["emb"].var(0) + 1e-6)
    FEATS[r] = np.argsort(ratio)[-4:][::-1]
    fig, axes = plt.subplots(4, 6, figsize=(12, 8.4))
    for i, d in enumerate(FEATS[r]):
        top = np.argsort(REP[r]["emb"][:, d])[-6:][::-1]
        for c, j in enumerate(top):
            axes[i, c].imshow(show_img(images[j])); axes[i, c].axis("off")
        axes[i, 0].set_title(f"feature {d}", fontsize=9, loc="left")
    fig.suptitle(f"{r}: features that changed most", fontsize=12)
    plt.tight_layout(); plt.show()

## 6. What do the features track?

Rank correlations between simple image measurements and (a) the features above, (b) the top three principal components (main axes) of each art-trained space. The sign of a principal component is arbitrary, so compare strengths, not signs, across models.

In [ ]:
P = np.array([image_props(im) for im in images])
rank = lambda M: np.column_stack([rankdata(c) for c in M.T])
fig, axes = plt.subplots(1, len(AFTERS), figsize=(5.8 * len(AFTERS), 5.6), squeeze=False, constrained_layout=True)
for ax, r in zip(axes[0], AFTERS):
    Fm = np.column_stack([REP[r]["emb"][:, FEATS[r]], PCA(3).fit_transform(REP[r]["emb"])])
    names = [f"feature {d}" for d in FEATS[r]] + ["PC1", "PC2", "PC3"]
    C = np.corrcoef(rank(Fm).T, rank(P).T)[:len(names), len(names):]
    im = ax.imshow(C, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(PROP_NAMES))); ax.set_xticklabels(PROP_NAMES, rotation=40, ha="right", fontsize=8)
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=8)
    for i in range(C.shape[0]):
        for j in range(C.shape[1]):
            ax.text(j, i, f"{C[i, j]:.2f}", ha="center", va="center", fontsize=6.5)
    ax.set_title(r, fontsize=10)
fig.colorbar(im, ax=axes[0].tolist(), label="Spearman r", shrink=0.8)
plt.show()

## 7. Where each model looks

Last-layer attention from the CLS token, averaged over heads, for one test painting per style. Columns: the painting, then every model before and after art training.

In [ ]:
LAST = -1
ATT_REPS = [r for r in REPS if REP[r]["att"] is not None]
fig, axes = plt.subplots(len(examples_pos), 1 + len(ATT_REPS),
                         figsize=(2.5 * (1 + len(ATT_REPS)), 2.6 * len(examples_pos)), squeeze=False)
for i, pos in enumerate(examples_pos):
    img = show_img(images[te_idx[pos]])
    axes[i, 0].imshow(img); axes[i, 0].set_title(class_names[labels[te_idx[pos]]], fontsize=9)
    for j, r in enumerate(ATT_REPS, start=1):
        a = REP[r]["att"][pos, LAST].astype(np.float32).mean(0)
        axes[i, j].imshow(img); axes[i, j].imshow(upsample(a), cmap="inferno", alpha=0.6)
        if i == 0:
            axes[i, j].set_title(r, fontsize=8)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout(); plt.show()

## 8. Attention per head (Baroque portrait)

For each setting: last-layer attention of every head before (top), after (middle) and the difference (bottom; red = gained). Titles give the before/after correlation r (1 = same) and total-variation distance TV (0 = identical, 1 = completely different).

In [ ]:
HEAD_POS = examples_pos[1] if len(examples_pos) > 1 else examples_pos[0]
for r in AFTERS:
    if REP[r]["att"] is None or before_of(r)["att"] is None:
        print(f"{r}: no attention maps"); continue
    A0 = before_of(r)["att"][HEAD_POS, LAST].astype(np.float32)
    A1 = REP[r]["att"][HEAD_POS, LAST].astype(np.float32)
    D = A1 - A0
    H, v = A0.shape[0], np.abs(D).max() + 1e-12
    fig, ax = plt.subplots(3, H + 1, figsize=(1.9 * (H + 1), 6))
    ax[0, 0].imshow(show_img(images[te_idx[HEAD_POS]])); ax[1, 0].axis("off"); ax[2, 0].axis("off")
    for h in range(H):
        rr = np.corrcoef(A0[h].ravel(), A1[h].ravel())[0, 1]
        tv = 0.5 * np.abs(D[h]).sum()
        ax[0, h + 1].imshow(A0[h], cmap="inferno"); ax[1, h + 1].imshow(A1[h], cmap="inferno")
        ax[2, h + 1].imshow(D[h], cmap="RdBu_r", vmin=-v, vmax=v)
        ax[0, h + 1].set_title(f"head {h}\nr={rr:.2f} TV={tv:.2f}", fontsize=7)
    for a in ax.ravel():
        a.set_xticks([]); a.set_yticks([])
    ax[0, 1].set_ylabel("before"); ax[1, 1].set_ylabel("after"); ax[2, 1].set_ylabel("after - before")
    fig.suptitle(r, fontsize=11); fig.tight_layout(); plt.show()

## 9. Attention change across all layers and heads

Mean change over all test paintings, on a shared colour scale. Frozen layers should be black.

In [ ]:
CHG = {}
for r in AFTERS:
    if REP[r]["att"] is not None and before_of(r)["att"] is not None:
        CHG[r] = 0.5 * np.abs(REP[r]["att"].astype(np.float32)
                              - before_of(r)["att"].astype(np.float32)).sum((3, 4)).mean(0)
if CHG:
    vmax = max(c.max() for c in CHG.values())
    fig, axes = plt.subplots(1, len(CHG), figsize=(4.4 * len(CHG), 4.8), squeeze=False, constrained_layout=True)
    for ax, (r, c) in zip(axes[0], CHG.items()):
        im = ax.imshow(c, cmap="magma", vmin=0, vmax=vmax, aspect="auto")
        L, H = np.unravel_index(c.argmax(), c.shape)
        ax.set_title(f"{r}\nmax {c.max():.3f} (layer {L}, head {H})", fontsize=9)
        ax.set_xlabel("head"); ax.set_ylabel("layer")
    fig.colorbar(im, ax=axes[0].tolist(), label="mean TV distance")
    plt.show()

## 10. Which layers changed?

Left: how much each block's weights moved. Right: how similar each layer's representation stays before vs after (CKA; 1 = unchanged).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.4))
for r in AFTERS:
    w = REP[r]["weights"]
    blocks = sorted((int(k.split()[1]), v) for k, v in w.items() if k.startswith("block "))
    axes[0].plot([b for b, _ in blocks], [v for _, v in blocks], marker="o", color=COL[r], label=r)
    other = {k: v for k, v in w.items() if not k.startswith("block ")}
    print(f"{r}: non-block parts " + ", ".join(f"{k} {v:.1e}" for k, v in other.items()))
    Lb, La = before_of(r)["layers"], REP[r]["layers"]
    if Lb is not None and La is not None:
        cka = [linear_cka(b, a) for b, a in zip(Lb, La)]
        axes[1].plot(range(len(cka)), cka, marker="o", color=COL[r], label=r)
axes[0].set_title("Relative weight change per block"); axes[0].set_xlabel("block")
axes[1].set_title("Representation similarity before vs after (CKA)"); axes[1].set_xlabel("layer output (0 = first)")
axes[1].set_ylim(0, 1.05)
for ax in axes:
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 11. Abstract vs representational

Split ratio for every representation with 95% bootstrap intervals. Human art experts treat abstract and representational art as *more alike* than novices do (Bimler et al., 2019), so an expert-like change would lower the ratio.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.6))
for i, r in enumerate(REPS):
    v = REP[r]["split"]; lo, hi = REP[r]["split_ci"]
    ax.bar(i, v, yerr=[[v - lo], [hi - v]], color=COL[r], capsize=5, alpha=1.0 if REP[r]["kind"] == "after" else 0.45)
ax.set_xticks(range(len(REPS))); ax.set_xticklabels(REPS, rotation=30, ha="right", fontsize=8)
ax.set_ylabel("split ratio"); ax.set_title("How separate are abstract and representational paintings?")
plt.tight_layout(); plt.show()

rng = np.random.default_rng(0)
for r in AFTERS:
    e0, e1 = before_of(r)["emb"], REP[r]["emb"]
    diffs = []
    for _ in range(500):
        ia, ir = rng.choice(ia_all, len(ia_all)), rng.choice(ir_all, len(ir_all))
        diffs.append(split_ratio(e1, ia, ir) - split_ratio(e0, ia, ir))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    print(f"{r}: change {REP[r]['split'] - before_of(r)['split']:+.3f}   95% CI [{lo:+.3f}, {hi:+.3f}]")

## 12. Symmetry (toy test)

How much more "art-like" asymmetric patterns look than symmetric ones (typicality = similarity to the average painting), with 95% intervals. Art experts prefer asymmetry (Leder et al., 2019); an expert-like shift would push the bar up after training. A crude proxy: treat as a prompt, not evidence.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.4))
for i, r in enumerate(REPS):
    s, a = REP[r]["typ"]["sym"], REP[r]["typ"]["asym"]
    d = a.mean() - s.mean()
    se = np.sqrt(a.var(ddof=1) / len(a) + s.var(ddof=1) / len(s))
    ax.bar(i, d, yerr=1.96 * se, color=COL[r], capsize=5, alpha=1.0 if REP[r]["kind"] == "after" else 0.45)
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks(range(len(REPS))); ax.set_xticklabels(REPS, rotation=30, ha="right", fontsize=8)
ax.set_ylabel("asymmetric − symmetric typicality"); ax.set_title("Do asymmetric patterns look more like art?")
plt.tight_layout(); plt.show()

## 13. How similar are the models' maps of art?

CKA between the final embeddings of every pair of representations, over all 960 paintings (1 = same geometry). Shows whether DINOv2 and DINOv3 start from similar views of art, and whether art training brings them closer together.

In [ ]:
M = np.array([[linear_cka(REP[a]["emb"], REP[b]["emb"]) for b in REPS] for a in REPS])
fig, ax = plt.subplots(figsize=(1.3 * len(REPS) + 3, 1.1 * len(REPS) + 2))
im = ax.imshow(M, cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(len(REPS))); ax.set_xticklabels(REPS, rotation=35, ha="right", fontsize=8)
ax.set_yticks(range(len(REPS))); ax.set_yticklabels(REPS, fontsize=8)
for i in range(len(REPS)):
    for j in range(len(REPS)):
        ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if M[i, j] < 0.6 else "black")
fig.colorbar(im, label="CKA"); ax.set_title("Similarity of embedding geometry")
plt.tight_layout(); plt.show()

## Reading across the results

- **What each model already knows:** section 1, "before" bars.
- **What art training adds versus reorganises:** probe accuracy (information) against silhouette and split ratio (organisation), sections 1–3.
- **Where in the network it happens:** sections 9–10.
- **Whether it looks expert-like:** sections 11–12; a lower split ratio and a higher asymmetry bar would be expert-like.
- **Whether DINOv2 and DINOv3 converge:** section 13.

Every figure uses seed 0; check section 1's error bars before reading much into small differences.